# 7.5 多 Rank 扩展性分析

## 本节学习目标

- 分析 collective 与端到端时间
- 避免错误性能归因

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 计时字段

`HCCL Communication` 统计 Broadcast/AllGather 及相应 stream synchronize；`Data Transfer` 统计 collective 前后的 H2D/D2H 搬移；本地 SpMV 由 Ascend C RTC Device kernel 执行，输出字段为 `Local SpMV launch-to-complete`（内部 `local_spmv_launch_to_complete_ms`，kernel 提交到 stream 同步完成）；`Total Time` 包含完整组织开销。旧历史表的 `HCCL collectives`/`ACL transfers`/`SpMV compute`/`Distributed time` 字段名只用于理解旧实现。

## 扩展性判断

rank 增加会减少局部行数，但完整 x Broadcast 和完整 y AllGather 不会同比缩小。小矩阵常被固定通信延迟主导，最优 rank 数依输入与拓扑而定。

## 预期现象与结果分析

不能把 total speedup 写成纯本地 SpMV speedup。正确结论应聚焦 communicator、数据划分、collective、transfer 以及本地 Device SpMV 的 launch-to-complete 与同步开销。

## 原工程 910B 历史实测（旧 Host-local + HCCL，非当前 Ascend C local SpMV）

课程副本 `src/hccl_spmv/README.md` 保留了原工程在单机 8 张 Ascend 910B、CANN 9.0、2/4/8 rank、`warmup=10`、`repeat=100` 下的 18 组记录，所有误差均为 0。该表来自整改前的 Host-local 版本：其 `SpMV compute` 列是旧 Host 计算的数值，不是当前 Ascend C Device kernel 实测。以下选取能说明趋势的配置：

<table style="margin-left: 0; text-align: left;">
  <thead>
    <tr>
      <th>Matrix</th>
      <th>Rank</th>
      <th>Distributed total (ms)</th>
      <th>HCCL (ms)</th>
      <th>SpMV compute (ms，旧 Host-local)</th>
      <th>Speedup</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>U1</td>
      <td>2</td>
      <td>3.999525</td>
      <td>1.518316</td>
      <td>1.783645</td>
      <td>0.812x</td>
    </tr>
    <tr>
      <td>U1</td>
      <td>4</td>
      <td>3.150330</td>
      <td>1.883229</td>
      <td>0.879784</td>
      <td>1.148x</td>
    </tr>
    <tr>
      <td>U1</td>
      <td>8</td>
      <td>7.489282</td>
      <td>6.030169</td>
      <td>0.694759</td>
      <td>0.679x</td>
    </tr>
    <tr>
      <td>U2</td>
      <td>8</td>
      <td>51.553612</td>
      <td>24.927603</td>
      <td>17.716824</td>
      <td>2.440x</td>
    </tr>
    <tr>
      <td>L2</td>
      <td>4</td>
      <td>47.474876</td>
      <td>11.057539</td>
      <td>29.618490</td>
      <td>2.133x</td>
    </tr>
  </tbody>
</table>

U1 从 4 rank 增至 8 rank 时局部计算继续下降，但 HCCL 增大使端到端性能回退；较大的 U2 能更好摊薄通信。该历史表生成于整改前的旧 Host-local 实现，只用于理解旧字段与趋势；当前实现（Ascend C RTC local SpMV + HCCL）的结论必须由当前真机运行重新生成。

## 课后实践

选择环境允许的 rank 集合，设计同一矩阵的扩展性记录表。

参考答案见 `answer/07.05_answer.md`。